# Generate Heating & Shape/Scale Files

Two tools:
1. **Heating file** (for `fixed_season` runs, or an optional diagnostic-only
   file for `gamma_ac` — not used by the running `gamma_ac` model).
2. **Shape/scale files** (for `gamma_ac` runs — these are what the model
   actually uses to draw stochastic daily heating). You can fit these to a
   plain climatology ("Control"), to composited event years like El Nino
   ("Composite"), or generate an explicit all-zero "No heating" pair.

Fill in the fields and click Generate. Output goes wherever you point
**Output dir** — usually your own preprocess directory, not the shared
instructor one (you likely don't have write access there anyway).


In [ ]:
import sys, os

def _find_project_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.exists(os.path.join(d, "scripts", "_config.py")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("Could not find the project root (looked for scripts/_config.py above "
                        + start + "). Make sure this notebook is somewhere inside the repo.")

PROJECT_ROOT = _find_project_root(os.getcwd())
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts"))
print("Project root:", PROJECT_ROOT)


## Part 1 — Heating File

In [ ]:
import ipywidgets as w
from generate_heating import generate_heating_file

heating_model_type = w.Dropdown(options=["fixed_season", "gamma_ac"], value="fixed_season", description="Model:")
heating_source = w.Dropdown(options=["custom", "cca", "cesm2", "era5"], value="custom", description="Source:")
heating_name = w.Text(value="", description="Name:", placeholder="e.g. MyScenario")
heating_season = w.Dropdown(options=["DJF", "JJA", "MAM", "SON"], value="DJF", description="Season:")
heating_start_year = w.IntText(value=1999, description="Start yr:")
heating_end_year = w.IntText(value=2020, description="End yr:")
heating_output_dir = w.Text(value="", description="Output dir:", placeholder="/path/to/your/preprocess/dir")
heating_file_path = w.Text(value="", description="Heating file:", placeholder="(for source=custom)")
heating_precip_path = w.Text(value="", description="Precip file:", placeholder="(for cca/cesm2/era5)")
heating_enso_years = w.Text(value="", description="ENSO years:", placeholder="e.g. 1998,2015,2016 (gamma_ac default only)")
heating_zw = w.Dropdown(options=[42, 63, 124], value=63, description="zw:")
heating_kmax = w.Dropdown(options=[11, 26], value=26, description="kmax:")

heating_output = w.Output()
heating_button = w.Button(description="Generate Heating File", button_style="primary")

FIXED_SOURCES = ["custom", "cca", "cesm2", "era5"]
GAMMA_SOURCES = ["custom", "default_enso_composite"]


def on_model_type_change(change):
    is_fixed = change["new"] == "fixed_season"
    heating_season.layout.display = "" if is_fixed else "none"
    heating_start_year.layout.display = "" if is_fixed else "none"
    heating_end_year.layout.display = "" if is_fixed else "none"
    heating_source.options = FIXED_SOURCES if is_fixed else GAMMA_SOURCES
    heating_source.value = heating_source.options[0]


def on_heating_source_change(change):
    is_custom = change["new"] == "custom"
    heating_file_path.layout.display = "" if is_custom else "none"
    heating_precip_path.layout.display = "none" if (is_custom or heating_model_type.value != "fixed_season") else ""
    heating_enso_years.layout.display = "none" if (is_custom or heating_model_type.value != "gamma_ac") else ""


heating_model_type.observe(on_model_type_change, names="value")
heating_source.observe(on_heating_source_change, names="value")
on_model_type_change({"new": heating_model_type.value})
on_heating_source_change({"new": heating_source.value})


def on_generate_heating_clicked(b):
    with heating_output:
        heating_output.clear_output()
        try:
            if not heating_name.value:
                raise ValueError("Name is required.")
            if not heating_output_dir.value:
                raise ValueError("Output dir is required.")

            kwargs = dict(
                model_type=heating_model_type.value,
                heating_source=("custom" if heating_source.value == "custom" else
                                 (heating_source.value if heating_model_type.value == "fixed_season"
                                  else "cmap_default")),
                heating_name=heating_name.value,
                output_dir=heating_output_dir.value,
                zw=heating_zw.value,
                kmax=heating_kmax.value,
            )
            if heating_model_type.value == "fixed_season":
                kwargs.update(season=heating_season.value,
                               start_year=heating_start_year.value,
                               end_year=heating_end_year.value)
            if heating_source.value == "custom":
                kwargs["heating_file"] = heating_file_path.value
            elif heating_model_type.value == "fixed_season":
                key = {"cesm2": "cesm2_precip_file", "cca": "cca_precip_file",
                       "era5": "era5_precip_file"}[heating_source.value]
                kwargs[key] = heating_precip_path.value
            elif heating_model_type.value == "gamma_ac" and heating_enso_years.value:
                kwargs["enso_warm_years"] = [y.strip() for y in heating_enso_years.value.split(",")]

            outfile = generate_heating_file(**kwargs)
            print(f"Done. Wrote: {outfile}")
        except Exception as e:
            print(f"ERROR: {e}")


heating_button.on_click(on_generate_heating_clicked)

heating_panel = w.VBox([
    w.HTML("<b>Heating File Generator</b>"),
    heating_model_type, heating_source, heating_name,
    heating_season, heating_start_year, heating_end_year,
    heating_file_path, heating_precip_path, heating_enso_years,
    heating_zw, heating_kmax, heating_output_dir,
    heating_button, heating_output,
])

display(heating_panel)


## Part 2 — Shape/Scale Files (Gamma_AC)

In [ ]:
import ipywidgets as w
from generate_shape_scale import fit_gamma_shape_scale, zero_shape_scale

ss_mode = w.Dropdown(
    options=["Control (fit to full period)", "Composite (e.g. El Nino years)", "No heating (zero)"],
    description="Mode:")
ss_name = w.Text(description="Name:")
ss_output_dir = w.Text(description="Output dir:")
ss_precip_glob = w.Text(
    value="/data/esplab/shared/obs/gridded/atm/precip/daily/CMORPH/CMORPH_V1.0_ADJ_0.25deg-DLY_00Z_*.nc",
    description="Precip glob:")
ss_precip_varname = w.Text(value="cmorph", description="Var name:")
ss_start_date = w.DatePicker(description="Start:")
ss_end_date = w.DatePicker(description="End:")
ss_scale_qc_max = w.FloatText(value=300.0, description="QC max:")
ss_zw = w.Dropdown(options=[42, 63, 124], value=63, description="zw:")

ss_windows_box = w.VBox([])
ss_add_window_btn = w.Button(description="+ Add window")


def make_window_row():
    s = w.DatePicker(description="Window start:")
    e = w.DatePicker(description="Window end:")
    rm = w.Button(description="Remove", button_style="danger", layout=w.Layout(width="80px"))
    row = w.HBox([s, e, rm])

    def on_remove(b):
        ss_windows_box.children = tuple(c for c in ss_windows_box.children if c is not row)

    rm.on_click(on_remove)
    return row


def on_add_window(b):
    ss_windows_box.children = ss_windows_box.children + (make_window_row(),)


ss_add_window_btn.on_click(on_add_window)

ss_output = w.Output()
ss_button = w.Button(description="Generate Shape/Scale Files", button_style="primary")


def on_mode_change(change):
    is_control = change["new"].startswith("Control")
    is_composite = change["new"].startswith("Composite")
    ss_precip_glob.layout.display = "" if (is_control or is_composite) else "none"
    ss_precip_varname.layout.display = "" if (is_control or is_composite) else "none"
    ss_start_date.layout.display = "" if (is_control or is_composite) else "none"
    ss_end_date.layout.display = "" if (is_control or is_composite) else "none"
    ss_scale_qc_max.layout.display = "" if is_composite else "none"
    ss_windows_box.layout.display = "" if is_composite else "none"
    ss_add_window_btn.layout.display = "" if is_composite else "none"


ss_mode.observe(on_mode_change, names="value")
on_mode_change({"new": ss_mode.value})


def on_generate_ss_clicked(b):
    with ss_output:
        ss_output.clear_output()
        try:
            if not ss_name.value:
                raise ValueError("Name is required.")
            if not ss_output_dir.value:
                raise ValueError("Output dir is required.")

            if ss_mode.value.startswith("No heating"):
                zero_shape_scale(ss_output_dir.value, ss_name.value, ss_zw.value)
            else:
                if not (ss_start_date.value and ss_end_date.value):
                    raise ValueError("Start/end date required.")
                windows = None
                if ss_mode.value.startswith("Composite"):
                    windows = []
                    for row in ss_windows_box.children:
                        s_widget, e_widget, _ = row.children
                        if not (s_widget.value and e_widget.value):
                            raise ValueError("Every composite window needs both a start and end date.")
                        windows.append((str(s_widget.value), str(e_widget.value)))
                    if not windows:
                        raise ValueError("Add at least one composite window, or switch to Control mode.")
                fit_gamma_shape_scale(
                    precip_glob=ss_precip_glob.value,
                    precip_varname=ss_precip_varname.value,
                    date_range=(str(ss_start_date.value), str(ss_end_date.value)),
                    zw=ss_zw.value,
                    output_dir=ss_output_dir.value,
                    name=ss_name.value,
                    composite_windows=windows,
                    scale_qc_max=ss_scale_qc_max.value if ss_mode.value.startswith("Composite") else None,
                )
            print("Done.")
        except Exception as e:
            print(f"ERROR: {e}")


ss_button.on_click(on_generate_ss_clicked)

ss_panel = w.VBox([
    w.HTML("<b>Shape/Scale File Generator (Gamma_AC)</b>"),
    ss_mode, ss_name, ss_output_dir, ss_precip_glob, ss_precip_varname,
    ss_start_date, ss_end_date, ss_scale_qc_max,
    ss_windows_box, ss_add_window_btn, ss_zw,
    ss_button, ss_output,
])

display(ss_panel)
